# Jobicy data collection and cleaning

This notebook collects all U.S.-eligible remote job postings from the Jobicy API (no role filter) and cleans them into the CSV files that are loaded into MySQL.

- Every API response is saved unchanged in a dated folder under `data/jobicy/raw/`, so later steps can always be rerun.
- Cleaning always reads those saved files, never the live API, so every cleaned row traces back to a saved response. Postings from every pull are combined, so the dataset grows with each pull.
- Jobicy asks API users to credit Jobicy and link each posting to its original page, so `job_url` is kept.

Run the cells from top to bottom. Set `COLLECT_NEW_DATA = 1` only when you want a new pull from the API.

Install packages (first run only)

In [ ]:
%pip install beautifulsoup4 requests

Settings

In [ ]:
import json
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup

# 1 = call the Jobicy API now and save a new dated raw folder
# 0 = skip collection and only clean the raw files already saved
COLLECT_NEW_DATA = 0

# 1 = write jobicy_all_jobs.csv and the cleaned CSV files
SAVE_FILES = 1

# Paths are relative to this notebook (notebooks/), so it runs on any computer
DATA_FOLDER = Path("..") / "data"
RAW_FOLDER = DATA_FOLDER / "jobicy" / "raw"
FLAT_RAW_FILE = DATA_FOLDER / "jobicy" / "jobicy_all_jobs.csv"
CLEANED_FOLDER = DATA_FOLDER / "cleaned"

# Jobicy API
url = "https://jobicy.com/api/jobs"

# Every search is limited to jobs open to U.S. applicants. There is no role filter.
GEO = "usa"

# Searches in one pull: first all U.S.-eligible jobs, then one search per Jobicy
# industry, which reaches postings the first search's pages may not include.
# Industry slugs from jobicy.com/categories (November 2026).
INDUSTRY_SLUGS = [
    "engineering",          # Software Engineering
    "data-science",         # Data Science & Analytics
    "cybersecurity",        # Cybersecurity
    "admin",                # DevOps & Infrastructure
    "supporting",           # Customer Support & Success
    "management",           # Product & Operations
    "project-management",   # Project & Program Management
    "design-multimedia",    # Creative & Design
    "marketing",            # Marketing & Sales
    "copywriting",          # Content & Editorial
    "admin-support",        # Admin & Virtual Assistance
    "accounting-finance",   # Finance & Accounting
    "hr",                   # HR & Recruiting
    "legal",                # Legal & Compliance
    "education",            # Education & E-learning
    "healthcare"            # Healthcare & Medical
]

# Jobicy asks for light use: no more than one pull per hour
MIN_MINUTES_BETWEEN_PULLS = 60
PAUSE_SECONDS = 3           # wait between requests inside a pull
PAGE_SIZE = 100             # postings per request (the API maximum)

# Role tags used by the first pull (31 Oct 2026). Collection no longer uses them;
# they are kept so the saved files from that pull are read the same way as before.
OLD_ROLE_TAGS = [
    "Data Analyst",
    "Data Scientist",
    "Data Engineer",
    "Software Engineer",
    "AI Engineer"
]

print("Raw folder found:", RAW_FOLDER.exists())

Step 0: Collect new postings from the Jobicy API

Runs only when `COLLECT_NEW_DATA = 1`. Each API response is saved unchanged in a new folder named by the UTC time of the pull, for example `data/jobicy/raw/2026-11-06T0115Z/all_us_jobs.json`. Earlier pulls are never overwritten, so postings build up over time.

One pull makes these searches, all with `geo=usa`:
1. all U.S.-eligible jobs (`all_us_jobs.json`)
2. one search per industry (`industry_<slug>.json`)

If a response says `hasMore`, the next page is requested with `nextCursor` and saved as `<name>_page2.json`, and so on. Jobicy asks that a new pull start no more than once per hour, so the cell refuses to start a pull within an hour of the last one.

In [ ]:
if COLLECT_NEW_DATA == 1:

    # Respect Jobicy's limit: no new pull within an hour of the last one
    now = datetime.now(timezone.utc)
    last_pull = None
    if RAW_FOLDER.exists():
        for folder in RAW_FOLDER.iterdir():
            if folder.is_dir():
                folder_time = datetime.strptime(folder.name, "%Y-%m-%dT%H%MZ").replace(tzinfo=timezone.utc)
                if last_pull is None or folder_time > last_pull:
                    last_pull = folder_time

    minutes_since_last = None
    if last_pull is not None:
        minutes_since_last = (now - last_pull).total_seconds() / 60

    if minutes_since_last is not None and minutes_since_last < MIN_MINUTES_BETWEEN_PULLS:
        print(f"Last pull was {minutes_since_last:.0f} minutes ago. Jobicy asks for at most one pull per hour; try again later.")

    else:
        # One folder per pull, named by the UTC time the pull started
        pull_folder = RAW_FOLDER / now.strftime("%Y-%m-%dT%H%MZ")
        pull_folder.mkdir(parents=True, exist_ok=True)

        # Each search: [file name, extra filters on top of geo]
        searches = [["all_us_jobs", {}]]
        for slug in INDUSTRY_SLUGS:
            searches.append(["industry_" + slug, {"industry": slug}])

        pull_job_ids = set()

        for name, filters in searches:

            cursor = None
            page = 1

            while True:

                params = {"geo": GEO, "count": PAGE_SIZE}
                for key in filters:
                    params[key] = filters[key]

                # Later pages are requested with the cursor from the previous response
                if cursor is not None:
                    params["cursor"] = cursor

                try:
                    response = requests.get(url, params=params, timeout=30)
                    response.raise_for_status()
                    data = response.json()
                except requests.RequestException as e:
                    print(name, "request failed:", e)
                    break

                if data.get("success") is False:
                    print(name, "API error:", data.get("error"))
                    break

                # Save the response exactly as received
                if page == 1:
                    out_file = pull_folder / f"{name}.json"
                else:
                    out_file = pull_folder / f"{name}_page{page}.json"

                with open(out_file, "w", encoding="utf-8") as file:
                    json.dump(data, file, indent=2)

                page_jobs = data.get("jobs", [])
                for job in page_jobs:
                    pull_job_ids.add(job.get("id"))

                note = ""
                if "industry" in filters and "industry" not in data.get("appliedFilters", {}):
                    note = "  (warning: the response does not list the industry filter)"

                print(f"{name:32s} page {page}: {len(page_jobs):3d} jobs{note}")

                time.sleep(PAUSE_SECONDS)

                if data.get("hasMore") and data.get("nextCursor"):
                    cursor = data["nextCursor"]
                    page = page + 1
                else:
                    break

        print("\nSaved new pull in folder:", pull_folder.name)
        print("Unique postings in this pull:", len(pull_job_ids))

else:
    print("Skipped collection (COLLECT_NEW_DATA = 0). Using the raw files already saved.")

Step 1: Load every saved raw response

Reads every pull folder under `data/jobicy/raw/`. Each record gets the search that returned it (`searched_role`) and the time of the pull (`collected_at`, read from the folder name). `searched_role` is the role tag for the first pull (31 Oct 2026), and `All US jobs` or `Industry: <slug>` for later pulls.

The flattened file `data/jobicy/jobicy_all_jobs.csv` keeps one row per posting per search per pull. Duplicates are removed in Step 2, not here.

In [ ]:
raw_records = []

pull_folders = sorted(p for p in RAW_FOLDER.iterdir() if p.is_dir())

for pull_folder in pull_folders:

    # Folder name is the UTC time of the pull, e.g. 2026-10-31T1852Z
    collected_at = datetime.strptime(pull_folder.name, "%Y-%m-%dT%H%MZ").replace(tzinfo=timezone.utc)

    records_before = len(raw_records)
    raw_files = sorted(pull_folder.glob("*.json"))

    for raw_file in raw_files:

        with open(raw_file, encoding="utf-8") as file:
            data = json.load(file)

        # Which search returned this file: a role tag (first pull), all jobs, or an industry
        search_name = raw_file.stem.split("_page")[0]
        tag = data.get("appliedFilters", {}).get("tag")

        if tag:
            searched_role = tag
        elif search_name.startswith("industry_"):
            searched_role = "Industry: " + search_name[len("industry_"):]
        else:
            searched_role = "All US jobs"

        for job in data.get("jobs", []):
            record = job.copy()
            record["searched_role"] = searched_role
            record["collected_at"] = collected_at
            raw_records.append(record)

    print(pull_folder.name, ":", len(raw_files), "files,", len(raw_records) - records_before, "records")

raw_jobs = pd.DataFrame(raw_records)

if raw_jobs.empty:
    raise ValueError("No raw Jobicy files found. Set COLLECT_NEW_DATA = 1 and run Step 0.")

if SAVE_FILES == 1:
    raw_jobs.to_csv(FLAT_RAW_FILE, index=False)

print("\nRaw records (posting x search x pull):", len(raw_jobs))
print("Unique Jobicy job IDs:", raw_jobs["id"].nunique())
print("\nRecords per search:")
print(raw_jobs["searched_role"].value_counts())

Step 2: Clean the postings

- One row per Jobicy job ID. When a posting comes back from several searches or pulls, the most recent copy is kept. `collected_at` is the first time it was seen and `last_seen_at` the last time.
- `searched_role` is the first search that returned the posting. It is not the role category, which comes from the job title (and, for unmatched titles, the industry) in MySQL.
- HTML tags are removed from descriptions, and extra spaces are removed from titles, company names and locations.
- Kept for the dashboard: `job_url` (link to the original posting), the salary range, `job_level` and `job_industry`.

In [ ]:
# Order of searches inside one pull, so "first search" is well defined:
# the old role tags, then all U.S. jobs, then the industries in INDUSTRY_SLUGS order
search_order = {}
for tag in OLD_ROLE_TAGS:
    search_order[tag] = len(search_order)
search_order["All US jobs"] = len(search_order)
for slug in INDUSTRY_SLUGS:
    search_order["Industry: " + slug] = len(search_order)

jobs = raw_jobs.copy()
jobs["search_order"] = jobs["searched_role"].map(search_order).fillna(len(search_order))
jobs = jobs.sort_values(["collected_at", "search_order"], kind="stable")

# First and last time each posting was seen across all pulls
first_seen = jobs.groupby("id")["collected_at"].min()
last_seen = jobs.groupby("id")["collected_at"].max()

# First search tag that returned each posting
first_role = jobs.drop_duplicates(subset=["id"], keep="first").set_index("id")["searched_role"]

# Most recent copy of each posting (title or salary can change between pulls)
jobicy_clean = jobs.drop_duplicates(subset=["id"], keep="last").copy()

jobicy_clean["searched_role"] = jobicy_clean["id"].map(first_role)
jobicy_clean["collected_at"] = jobicy_clean["id"].map(first_seen)
jobicy_clean["last_seen_at"] = jobicy_clean["id"].map(last_seen)

# Salary fields are missing from a response when no posting in it lists pay
for column in ["salaryMin", "salaryMax", "salaryCurrency", "salaryPeriod"]:
    if column not in jobicy_clean.columns:
        jobicy_clean[column] = pd.NA

# Rename columns
jobicy_clean = jobicy_clean.rename(columns={
    "id": "job_id",
    "jobTitle": "job_title",
    "companyName": "company_name",
    "jobGeo": "location",
    "jobDescription": "description",
    "pubDate": "publication_date",
    "url": "job_url",
    "jobLevel": "job_level",
    "jobIndustry": "job_industry",
    "salaryMin": "salary_min",
    "salaryMax": "salary_max",
    "salaryCurrency": "salary_currency",
    "salaryPeriod": "salary_period"
})

# Remove HTML tags
def clean_html(text):

    if pd.isna(text):
        return ""

    soup = BeautifulSoup(str(text), "html.parser")

    return " ".join(soup.stripped_strings)

jobicy_clean["description_clean"] = (
    jobicy_clean["description"].apply(clean_html)
)

# Jobicy returns the industry as a list; so far every posting has exactly one
def industry_text(value):

    if isinstance(value, list):
        return "; ".join(value)

    if pd.isna(value):
        return ""

    return str(value)

jobicy_clean["job_industry"] = jobicy_clean["job_industry"].apply(industry_text)

# Remove extra spaces ("Canada,  USA" becomes "Canada, USA")
for column in ["job_title", "company_name", "location", "job_level", "job_url"]:
    jobicy_clean[column] = (
        jobicy_clean[column]
        .astype("string")
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

# Salaries as numbers; postings without pay stay empty
jobicy_clean["salary_min"] = pd.to_numeric(jobicy_clean["salary_min"], errors="coerce")
jobicy_clean["salary_max"] = pd.to_numeric(jobicy_clean["salary_max"], errors="coerce")

# Convert publication dates
jobicy_clean["publication_date"] = (
    pd.to_datetime(
        jobicy_clean["publication_date"],
        errors="coerce",
        utc=True
    )
)

# Columns saved to jobicy_jobs_clean.csv. The first six keep their original order.
output_columns = [
    "job_id",
    "job_title",
    "company_name",
    "location",
    "description_clean",
    "publication_date",
    "job_url",
    "job_level",
    "job_industry",
    "salary_min",
    "salary_max",
    "salary_currency",
    "salary_period",
    "searched_role",
    "collected_at",
    "last_seen_at"
]

jobicy_clean = jobicy_clean[output_columns + ["description"]].reset_index(drop=True)

print("Total Unique Jobs:", len(jobicy_clean))
print("With a salary range:", jobicy_clean["salary_min"].notna().sum())
print("With a job URL:", jobicy_clean["job_url"].notna().sum())
print("\nJob level:")
print(jobicy_clean["job_level"].value_counts())
print("\nIndustry:")
print(jobicy_clean["job_industry"].value_counts())

display(
    jobicy_clean[
        [
            "job_id",
            "job_title",
            "company_name",
            "location",
            "job_level",
            "salary_min",
            "salary_max"
        ]
    ].head()
)

Step 3: Extract technical skills from Jobicy

This is still the starting dictionary of 10 skills.

In [ ]:
import re

# Initial skill dictionary
skill_patterns = {
    "Python": r"\bpython\b",
    "SQL": r"\bsql\b",
    "Tableau": r"\btableau\b",
    "Power BI": r"\bpower\s*bi\b",
    "Microsoft Excel": r"\b(?:microsoft excel|ms excel)\b",
    "AWS": r"\b(?:aws|amazon web services)\b",
    "PostgreSQL": r"\b(?:postgresql|postgres)\b",
    "Snowflake": r"\bsnowflake\b",
    "Docker": r"\bdocker\b",
    "Java": r"\bjava\b"
}

# Extract skills from job descriptions
def extract_skills(description):

    found_skills = []

    for skill, pattern in skill_patterns.items():

        if re.search(
            pattern,
            description,
            flags=re.IGNORECASE
        ):
            found_skills.append(skill)

    return found_skills


jobicy_clean["extracted_skills"] = (
    jobicy_clean["description_clean"].apply(
        extract_skills
    )
)

display(
    jobicy_clean[
        [
            "job_title",
            "extracted_skills"
        ]
    ].head(10)
)

Step 4: Prepare the Job-Skills table

In [ ]:
# Create job-skill relationships
job_skills_clean = jobicy_clean[
    [
        "job_id",
        "extracted_skills"
    ]
].explode("extracted_skills")

# Rename column
job_skills_clean = job_skills_clean.rename(
    columns={
        "extracted_skills": "skill_name"
    }
)

# Remove rows without extracted skills
job_skills_clean = job_skills_clean.dropna(
    subset=["skill_name"]
)

# Remove duplicate job-skill pairs
job_skills_clean = job_skills_clean.drop_duplicates(
    subset=["job_id", "skill_name"]
)

print(
    "Total Job-Skill Relationships:",
    len(job_skills_clean)
)

display(job_skills_clean.head(20))

Step 5: Save the cleaned datasets

In [ ]:
if SAVE_FILES == 1:

    CLEANED_FOLDER.mkdir(parents=True, exist_ok=True)

    # Store only scalar columns to avoid CSV list serialization issues
    jobicy_clean[output_columns].to_csv(
        CLEANED_FOLDER / "jobicy_jobs_clean.csv",
        index=False
    )

    # Save extracted job-skill relationships
    job_skills_clean.to_csv(
        CLEANED_FOLDER / "job_skills_clean.csv",
        index=False
    )

    print("Saved", len(jobicy_clean), "postings and", len(job_skills_clean), "job-skill links")

else:
    print("Not saved (SAVE_FILES = 0)")